# Phase 3 — Exploratory Data Analysis
**Smart City Mobility Intelligence System**

Analyzing 500K+ ride records across Delhi NCR, Mumbai, Bengaluru, Chennai, and Hyderabad.

### Key Questions
1. When is demand highest — by hour, day, month?
2. Which cities have the worst wait times and highest surge?
3. How does weather affect rides, fares, and cancellations?
4. What is the impact of festivals and IPL match days?
5. Which vehicle types are most popular and cost-effective?
6. Which zones are hotspots for high wait times and surge?
7. What features correlate most with fare and wait time?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
})

df = pd.read_csv('../data/processed/rides_clean.csv', parse_dates=['timestamp'])
print(f'Dataset: {len(df):,} rows × {df.shape[1]} columns')
print(f'Date range: {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
df.head(3)

## 1. Dataset overview

In [ ]:
print('=== CITY DISTRIBUTION ===')
print(df['city'].value_counts().to_string())

print('\n=== VEHICLE TYPE SPLIT ===')
print(df['vehicle_type'].value_counts().to_string())

print('\n=== KEY STATISTICS ===')
df[['wait_time_min','surge_multiplier','distance_km','fare_inr','driver_rating']].describe().round(2)

## 2. Hourly demand patterns

In [ ]:
from IPython.display import Image
Image('../reports/figures/01_hourly_demand.png')

In [ ]:
hourly = df.groupby('hour').size()
peak_pct = hourly[[7,8,9,17,18,19,20]].sum() / hourly.sum() * 100
dead_pct = hourly[[1,2,3,4]].sum() / hourly.sum() * 100
print(f'Peak hours (7-9AM, 5-8PM) account for {peak_pct:.1f}% of all rides')
print(f'Dead hours (1-4AM) account for only {dead_pct:.1f}% of rides')
print(f'Evening peak (6PM) is the busiest single hour: {hourly[18]:,} rides')

## 3. City comparison

In [ ]:
Image('../reports/figures/02_city_comparison.png')

In [ ]:
city_stats = df.groupby('city').agg(
    rides=('ride_id','count'),
    median_wait=('wait_time_min','median'),
    median_fare=('fare_inr','median'),
    mean_surge=('surge_multiplier','mean'),
    completion_rate=('is_completed','mean')
).round(2)
city_stats['completion_rate'] = (city_stats['completion_rate'] * 100).round(1)
print(city_stats.to_string())

## 4. Weather impact

In [ ]:
Image('../reports/figures/03_weather_impact.png')

In [ ]:
weather_stats = df.groupby('weather').agg(
    rides=('ride_id','count'),
    median_wait=('wait_time_min','median'),
    mean_surge=('surge_multiplier','mean'),
    cancel_rate=('is_completed', lambda x: round((1-x.mean())*100, 1))
).round(2)
print(weather_stats)

heavy = weather_stats.loc['heavy_rain']
clear = weather_stats.loc['clear']
print(f'\nHeavy rain vs clear:')
print(f'  Wait time: {clear.median_wait}min → {heavy.median_wait}min (+{((heavy.median_wait/clear.median_wait)-1)*100:.0f}%)')
print(f'  Surge:     {clear.mean_surge:.2f}× → {heavy.mean_surge:.2f}× (+{((heavy.mean_surge/clear.mean_surge)-1)*100:.0f}%)')
print(f'  Cancels:   {clear.cancel_rate}% → {heavy.cancel_rate}%')

## 5. Vehicle type analysis

In [ ]:
Image('../reports/figures/04_vehicle_analysis.png')

## 6. Monthly trends & seasonality

In [ ]:
Image('../reports/figures/05_monthly_trends.png')

In [ ]:
season_stats = df.groupby('season').agg(
    rides=('ride_id','count'),
    mean_surge=('surge_multiplier','mean'),
    mean_wait=('wait_time_min','mean'),
    rain_pct=('is_raining','mean')
).round(3)
season_stats['rain_pct'] = (season_stats['rain_pct']*100).round(1)
print(season_stats)

## 7. Festival & IPL event impact

In [ ]:
Image('../reports/figures/06_event_impact.png')

## 8. Demand heatmap — hour × day of week

In [ ]:
Image('../reports/figures/07_demand_heatmap.png')

## 9. Feature correlation matrix

In [ ]:
Image('../reports/figures/08_correlation_heatmap.png')

In [ ]:
num_cols = ['wait_time_min','surge_multiplier','distance_km','fare_inr',
            'is_peak_hour','is_raining','is_festival','is_ipl_day','demand_pressure']
corr_fare = df[num_cols].corr()['fare_inr'].drop('fare_inr').sort_values(ascending=False)
print('Top correlates with fare_inr:')
print(corr_fare.round(3).to_string())

corr_wait = df[num_cols].corr()['wait_time_min'].drop('wait_time_min').sort_values(ascending=False)
print('\nTop correlates with wait_time_min:')
print(corr_wait.round(3).to_string())

## 10. Zone-level hotspots

In [ ]:
Image('../reports/figures/10_zone_hotspots.png')

## Key Findings Summary

| Finding | Metric |
|---|---|
| Evening peak (6PM) is busiest hour | 56,264 rides |
| Mumbai has highest median wait | 8.2 min |
| Heavy rain increases wait time | +84% vs clear |
| Heavy rain increases surge | +84% vs clear |
| Festival days increase fare | +60% vs normal |
| Monsoon is highest surge season | 1.71× avg |
| Surge is top fare predictor | r = 0.85 |
| 25.1% of rides have high surge (≥2×) | 125K+ rides |
| Peripheral zones wait 40% longer | Ghaziabad, Tambaram |
| IPL nights (9-11PM) surge 3× | Post-match spike |

---
**Next step → Phase 4: Demand forecasting ML model**